# «Қайда?» — дообучение детектора отходов (Kaggle / Colab, GPU)

ТЗ, раздел 6.2: YOLO (Ultralytics, n/s, веса COCO) → TACO + TrashNet + Roboflow Universe →
наши 13 классов → train/val/test 80/10/10 → аугментации (освещение, размытие, снег/ночь) →
fine-tuning → экспорт ONNX → проверка скорости на CPU.

**Kaggle:** Settings → Accelerator → GPU T4 ×2, Internet → On. **Colab:** Среда → GPU.
Результат — `model.onnx` + `model.json` в /kaggle/working (или /content). Загрузите его в
GitHub Release репозитория (тег `model-<версия>`), и ИИ-сервис подхватит его через `MODEL_URL`.

Лицензии: Ultralytics — AGPL-3.0 (для хакатона подходит); TACO — CC BY 4.0; TrashNet — MIT;
датасеты Roboflow — смотрите лицензию каждого.

In [ ]:
import os, sys, subprocess, pathlib
REPO = 'https://github.com/KhursandAtajanovAI/orleu-hack-2026-elim-ai.git'
BRANCH = 'main'
VERSION = 'waste-yolo-0.1'      # имя версии модели — попадёт в каждый ответ сервиса
BASE = 'yolo11n.pt'             # n — быстрее на CPU; s — точнее
EPOCHS = 80
WORK = pathlib.Path('/kaggle/working' if os.path.exists('/kaggle') else '/content')
os.chdir(WORK)
!git clone -q --depth 1 -b $BRANCH $REPO kaida
!pip -q install -r kaida/ai/requirements-train.txt

## 1. Датасеты

In [ ]:
# TACO (~1500 фото, 60 категорий) — скачивание по ссылкам из annotations.json
!git clone -q --depth 1 https://github.com/pedropro/TACO.git raw/TACO
!cd raw/TACO && pip -q install -r requirements.txt 2>/dev/null; python download.py --dataset_path ./data/annotations.json

In [ ]:
# TrashNet (2527 фото, 6 классов)
!mkdir -p raw && cd raw && wget -q -O trashnet.zip https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip && unzip -q -o trashnet.zip -d trashnet
TRASHNET = next(p for p in pathlib.Path('raw/trashnet').rglob('cardboard')).parent
print(TRASHNET)

In [ ]:
# Roboflow Universe (необязательно): нужен ключ — Kaggle → Add-ons → Secrets → ROBOFLOW_API_KEY
# Подберите наборы с плёнкой, тетрапаком, батарейками, текстилем — их мало в TACO/TrashNet.
RF = [  # (workspace, project, version)
    # ('roboflow-universe-projects', 'garbage-classification-3', 2),
]
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['ROBOFLOW_API_KEY'] = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    pass
if RF and os.environ.get('ROBOFLOW_API_KEY'):
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
    for ws, proj, ver in RF:
        rf.workspace(ws).project(proj).version(ver).download('yolov8', location=f'raw/rf_{proj}')

## 2. Маппинг к нашим классам, объединение, сплит 80/10/10, аугментации снег/ночь

In [ ]:
P = 'kaida/ai/training/prepare.py'
!python $P taco --src raw/TACO/data --out data/work
!python $P trashnet --src {TRASHNET} --out data/work
for d in pathlib.Path('raw').glob('rf_*'):
    !python $P yolo --src {d} --out data/work --name {d.name}
# свои фото Астаны (экспорт YOLO из CVAT/Label Studio) — только для проверки качества:
if pathlib.Path('raw/astana_yolo').exists():
    !python $P astana --src raw/astana_yolo --out data/work
!python $P split --work data/work --out data/merged
!python kaida/ai/training/augment_weather.py --data data/merged --fraction 0.25

## 3. Дообучение (GPU)

In [ ]:
!python kaida/ai/training/train.py --data data/merged/data.yaml --model $BASE --epochs $EPOCHS --device 0 --name $VERSION

## 4. Экспорт ONNX и проверка скорости на CPU (≤ 3 с на 1280 px)

In [ ]:
best = sorted(pathlib.Path('runs').rglob('best.pt'))[-1]
!python kaida/ai/training/export.py --weights {best} --version $VERSION --out {WORK}/model.onnx
!cd kaida/ai && CUDA_VISIBLE_DEVICES= python training/bench.py {WORK}/model.onnx --runs 20 --out {WORK}/bench.json
if pathlib.Path('data/merged/data_astana.yaml').exists():
    from ultralytics import YOLO
    m = YOLO(str(best)).val(data='data/merged/data_astana.yaml', split='test')
    print('mAP50 на фото Астаны:', m.box.map50)

## 5. Публикация модели

1. Скачайте `model.onnx`, `model.json`, `bench.json` из Output.
2. GitHub → Releases → *Draft a new release* → тег `model-waste-yolo-0.1` → прикрепите файлы.
3. В переменных ИИ-сервиса: `MODEL_URL=https://github.com/<owner>/<repo>/releases/download/model-waste-yolo-0.1/model.onnx`,
   `CLASS_MAP=native`. Или запустите workflow **AI · модель** с этим URL — он соберёт образ с моделью внутри.